# Notebook 02: Classification Heads on Frozen Encoder

Goal of this notebook: Evaluate different classification head architectures on top of the Moirai foundation model. 

We evaluate four distinct architectures:
1. Mean Pooling
2. Single-Scale Attention
3. Single-Scale Multi-Head Attention (MHA)
4. Hierarchical Multi-Head Attention

In [ ]:
import torch
import pandas as pd
from IPython.display import display

from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder
from heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from utils import get_lsst_dataloaders, get_z_loaders, grid_search_heads

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
HEADS_TO_TEST = [8, 16, 32, 64, 128, 384] 

# Hyperparameters
MOIRAI_BATCH_SIZE = 32
HEAD_BATCH_SIZE = 128
LR_GRID = [1e-3, 1e-4]
WD_GRID = [0.01, 0.05]
MODES = ["shared_context", "independent_context"]

# Results storage
df_mean = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES_TO_TEST)
df_attn = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_mha = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_mha8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_mha16 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_hier = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_tr, raw_va, raw_te, num_classes = get_lsst_dataloaders(batch_size=MOIRAI_BATCH_SIZE, device=DEVICE)

## The Fast Evaluation Loop
Since Moirai is frozen, we loop over the patch sizes, instantiate Moirai, and **pre-extract the embeddings ($Z$)**. We then pass these $Z$ loaders to the different head architectures.

In [3]:
# We define a function to run the evaluation for a specific patch size
def evaluate_heads_for_patch(patch):
    
    # 1. Instantiate the Frozen Encoder
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=patch, context_length=36, patch_size=patch, 
        num_samples=100, target_dim=NUM_VARS, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    )
    
    # 2. Pre-Extract Embeddings (Z)
    tr_z, va_z, te_z = get_z_loaders(
        encoder, raw_tr, raw_va, raw_te, 
        head_batch_size=HEAD_BATCH_SIZE, device=DEVICE, remove_last_patch=False, num_vars=NUM_VARS
    )
    
    return tr_z, va_z, te_z

## 1. Mean Pooling Architecture


How it works: The simplest baseline. It takes the contextual patches generated by Moirai and averages them over the temporal dimension. The result is a single flattened vector containing the average representation of each variable, which is then passed to a Linear classifier.

In [4]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch) # Extract once
    
    _, acc = grid_search_heads(
        MeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes},
        tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
    )
    df_mean.loc["Mean Pooling", patch] = acc

display(df_mean.astype(float).round(4))

,8,16,32,64
Mean Pooling,0.6156,0.6156,0.6071,0.6363


## 2. Single-Scale Attention Architecture


How it works: Instead of a naive average, this head uses a learned Query vector. This Query looks at all the patches (Keys/Values) and assigns an attention weight to each. The network learns which patches are the most important for the classification task.
* shared_context: One single Query acts across all variables simultaneously.
* independent_context: Each variable has its own dedicated Query to find its own important patches.

In [5]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    
    for mode in MODES:
        _, acc = grid_search_heads(
            SingleScaleAttentionClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        df_attn.loc[mode, patch] = acc

display(df_attn.astype(float).round(4))

,8,16,32,64
shared_context,0.6010,0.6038,0.5864,0.6350
independent_context,0.5965,0.6006,0.5908,0.6237


## 3. Multi-Head Attention (MHA)


How it works: A single attention mechanism might focus entirely on one feature (e.g., the highest peak). Multi-Head Attention solves this by projecting the data into multiple sub-spaces (e.g., 16 heads).
This allows the model to capture multiple distinct temporal concepts simultaneously (e.g., Head 1 looks at recent patches, Head 2 looks at periodic drops, ...).

### 3.1 MHA with 16 heads over all patch sizes

In [6]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    
    for mode in MODES:
        _, acc = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 16},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        df_mha.loc[mode, patch] = acc

display(df_mha.astype(float).round(4))

,8,16,32,64
shared_context,0.6083,0.6217,0.6006,0.6249
independent_context,0.6144,0.6217,0.6054,0.6249


### 3.2 MHA over patches of sizes 8 and 16 but with highter number of heads

In [10]:
for heads in HEADS_TO_TEST:
    for patch in [8,16]:
        tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
        
        for mode in MODES:
            _, acc = grid_search_heads(
                SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
            )
            if patch == 8:
                df_mha8.loc[mode, heads] = acc
            else:
                df_mha16.loc[mode, heads] = acc

In [11]:
display(df_mha8.astype(float).round(4))
display(df_mha16.astype(float).round(4))

,8,16,32,64,128,384
shared_context,0.6091,0.6184,0.6107,0.6200,0.6095,0.6103
independent_context,0.6135,0.6184,0.6188,0.6099,0.6111,0.6103


,8,16,32,64,128,384
shared_context,0.6127,0.6192,0.6208,0.6229,0.6261,0.6334
independent_context,0.6188,0.6217,0.6188,0.6208,0.6269,0.6273


## 4. Hierarchical Multi-Head Attention


How it works: Inspired by Hierarchical Attention Networks (HAN). It performs attention in two steps:
1. Temporal Attention: MHA is applied *within* each variable individually to summarize its temporal patches into a single variable-vector.
2. Variable Attention: A second MHA is applied *across* the summarized variables. This allows the network to learn inter-variable correlations.

In [ ]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    
    for mode in MODES:
        _, acc = grid_search_heads(
            HierarchicalMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 16},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        df_hier.loc[mode, patch] = acc

display(df_hier.astype(float).round(4))

,8,16,32,64
shared_context,0.5856,0.5868,0.5758,0.5884
independent_context,0.6030,0.5916,0.5754,0.5831


## Final Summary
Review of all architectures evaluated on the frozen Moirai encoder.

In [12]:
print("\n RESULTS: MEAN POOLING")
display(df_mean.astype(float).round(4))

print("\n RESULTS: ATTENTION (Base)")
display(df_attn.astype(float).round(4))

print("\n RESULTS: MULTI-HEAD ATTENTION")
display(df_mha.astype(float).round(4))

print("\n RESULTS: MULTI-HEAD ATTENTION (Patch = 8)")
display(df_mha8.astype(float).round(4))

print("\n RESULTS: MULTI-HEAD ATTENTION (Patch = 16)")
display(df_mha8.astype(float).round(4))

print("\n RESULTS: HIERARCHICAL MHA")
display(df_hier.astype(float).round(4))


 RESULTS: MEAN POOLING


,8,16,32,64
Mean Pooling,0.6156,0.6156,0.6071,0.6363



 RESULTS: ATTENTION (Base)


,8,16,32,64
shared_context,0.6010,0.6038,0.5864,0.6350
independent_context,0.5965,0.6006,0.5908,0.6237



 RESULTS: MULTI-HEAD ATTENTION


...

,8,16,32,64,128,384
shared_context,0.6091,0.6184,0.6107,0.6200,0.6095,0.6103
independent_context,0.6135,0.6184,0.6188,0.6099,0.6111,0.6103



 RESULTS: MULTI-HEAD ATTENTION (Patch = 16)


,8,16,32,64,128,384
shared_context,0.6091,0.6184,0.6107,0.6200,0.6095,0.6103
independent_context,0.6135,0.6184,0.6188,0.6099,0.6111,0.6103



 RESULTS: HIERARCHICAL MHA


,8,16,32,64
shared_context,0.5856,0.5868,0.5758,0.5884
independent_context,0.6030,0.5916,0.5754,0.5831
